# SuSiE colocalisation & fine-mapping for seven replicated genomic associations of myalgic encephalomyelitis/chronic fatigue syndrome (ME/CFS)

This notebook provides the code used to fine-map, and then colocalise, the replicated
ME/CFS GWAS loci with the [`susieR`](https://stephenslab.github.io/susieR/) and
[`coloc`](https://chr1swallace.github.io/coloc/) packages:

1. **Fine-mapping** — resolves the GWAS signal at each locus into credible sets of
   candidate causal variants (SuSiE), and creates the final figure seen in publication
   (association + posterior inclusion probability + gene track).
2. **Colocalisation** — for each locus, tests whether the GWAS association and a nearby
   eQTL share the same causal variant (`coloc.susie`), reported as the posterior
   probability of a shared signal (**PP.H4**).


### Section map
| Section | Contents |
|---|---|
| 1 · Environment & configuration | install packages, load libraries, set paths, unpack data |
| 2 · GWAS summary statistics & reference-panel LD | allele orientation, z-scores, LD matrices |
| 3 · Colocalisation (GWAS × eQTL) | SuSiE + `coloc.susie` across all eQTL datasets |
| 4 · Fine-mapping the GWAS signal | SuSiE fine-mapping + per-locus figures |

### Repository layout
This notebook is self-contained and uses **relative paths only** — run it from the
project root. The summary statistics are bundled as **zip archives** and unzipped into
`data/` by the last cell of Section 1; the genotype LD panels are **not** included and
must be supplied by the user. If one uses the 1000 Genomes reference panel for the UKB
summary statistics, do not expect to reproduce the results from the manuscript exactly.

```
.
├── susie_finemapping_coloc.ipynb
├── data/
│   ├── gwas.zip     ──unzip──▶  data/gwas/  replication_imputed_hg19.csv   (~14 MB)
│   ├── eqtl.zip     ──unzip──▶  data/eqtl/  *.tsv                          (~26 MB)
│   └── ld_panels/   (NOT included — provide your own)
│       ├── gwas_ukb/    ukb_wb_<locus>.{bed,bim,fam}   (UK Biobank; ~28 GB total)
│       └── eqtl_1000g/  EUR.{bed,bim,fam}              (1000G EUR; ~1.4 GB)
└── results/                                            (all outputs; prefix coloc_replication_*)
```

The two archives total ~10 MB in the repository and expand to **~40 MB** after extraction.

### Data availability
The **summary statistics are included** here as compressed files (`data/gwas.zip`,
`data/eqtl.zip`) and are extracted automatically on first run. The **genotype LD
reference panels are not included**: the GWAS-side panels are derived from UK Biobank
individual-level data, available only to approved researchers under the UK Biobank Access
Management System, and the eQTL-side panel is the 1000 Genomes EUR reference. 
The eQTL summary statistics were taken from GTEx v10, Jerber *et al.* (iPSC-derived
neurons) and MetaBrain.

- **GTEx v10** — [GTEx Portal](https://gtexportal.org/). Cite: GTEx Consortium.
  The GTEx Consortium atlas of genetic regulatory effects across human tissues.
  *Science* **369**, 1318–1330 (2020).
  [doi:10.1126/science.aaz1776](https://doi.org/10.1126/science.aaz1776)
- **iPSC-derived neurons** — Jerber, J., Seaton, D. D., Cuomo, A. S. E. *et al.*
  Population-scale single-cell RNA-seq profiling across dopaminergic neuron
  differentiation. *Nature Genetics* **53**, 304–312 (2021).
  [doi:10.1038/s41588-021-00801-6](https://doi.org/10.1038/s41588-021-00801-6)
- **MetaBrain** — de Klein, N., Tsai, E. A., Vochteloo, M. *et al.* Brain expression
  quantitative trait locus and network analyses reveal downstream effects and putative
  drivers for brain-related diseases. *Nature Genetics* **55**, 377–388 (2023).
  [doi:10.1038/s41588-023-01300-6](https://doi.org/10.1038/s41588-023-01300-6)
  · [metabrain.nl](https://www.metabrain.nl/)

### Requirements
R with `data.table`, `susieR`, `coloc` (≥ 5.1.0) and `jsonlite`, plus a
[PLINK 2](https://www.cog-genomics.org/plink/2.0/) executable on `PATH` (or set the
`PLINK2` environment variable). Gene tracks are fetched from the Ensembl REST API, so an
internet connection is needed for the fine-mapping figures.

---

**Analysis notes**

*Allele orientation.* The GWAS effect estimates are **not** harmonised to any
bim/reference. The TMLE effect (`beta1`) is the risk difference for the transition
Control genotype (homozygous for the most frequent allele) → Case genotype (adds the
less frequent allele). The GWAS **effect allele is therefore the minor allele**,
determined per variant from the observed genotype counts (robust to array label swaps).
The minor-allele-oriented z-score is aligned to each eQTL's effect allele (ALT),
flipping sign as needed, before colocalisation.

*Variant matching & LD.* Variants are matched by rsID (build-agnostic: GWAS + Jerber +
MetaBrain are GRCh37, GTEx is GRCh38, but all carry rsIDs). GWAS-side LD comes from a
per-locus UKB white-British panel; eQTL-side LD from a 1000G EUR panel (a European proxy
for the eQTL cohorts). In both panels the effect allele is forced as the reference
allele so the LD sign matches the z-score coding.


## 1 · Environment & configuration

Install the required R packages, load them, set all file paths and analysis constants,
and unpack the bundled summary-statistic archives into `data/`. `coloc` ≥ 5.1.0 is required 
for the `coloc.susie` function used in Section 3.


In [ ]:
## ---- install dependencies (coloc >= 5.1.0 for coloc.susie) --------------
repos <- "https://cloud.r-project.org"
pkgs  <- c("data.table", "susieR", "coloc")
new   <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(new)) install.packages(new, repos = repos)
if (packageVersion("coloc") < "5.1.0") {
  if (!requireNamespace("remotes", quietly = TRUE)) install.packages("remotes", repos = repos)
  remotes::install_github("chr1swallace/coloc")
}
cat(sprintf("coloc %s | susieR %s | data.table %s\n",
            packageVersion("coloc"), packageVersion("susieR"), packageVersion("data.table")))


In [ ]:
library(data.table)
library(susieR)
library(coloc)

In [ ]:
## ---- paths, panels and analysis constants -------------------------------
## All paths are RELATIVE to this notebook's directory (the project root); run the
## notebook from that directory. The genotype LD panels are individual-level data and
## are NOT distributed with this repository -- provide your own copies under
## data/ld_panels/ (see the .gitignore and README).
DATA_DIR    <- "data"
GWAS_FILE   <- file.path(DATA_DIR, "gwas", "replication_imputed_hg19.csv")  # GRCh37 GWAS summary statistics
EQTL_DIR    <- file.path(DATA_DIR, "eqtl")                                  # prepared eQTL summary statistics (TSV)
LD_GWAS_DIR <- file.path(DATA_DIR, "ld_panels", "gwas_ukb")                 # GWAS-side LD panels (per locus, PLINK bfiles)
KG_EQTL     <- file.path(DATA_DIR, "ld_panels", "eqtl_1000g", "EUR")        # eQTL-side LD panel (1000G EUR) bfile prefix
PLINK2      <- Sys.getenv("PLINK2", "plink2")                               # PLINK 2 executable (on PATH, or set $PLINK2)

## GWAS-side LD panel per coloc locus / chromosome (PLINK bfile prefixes)
KG_BFILES   <- c("12" = file.path(LD_GWAS_DIR, "ukb_wb_bicd1_dnm1l"),
                 "13" = file.path(LD_GWAS_DIR, "ukb_wb_clybl"),
                 "16" = file.path(LD_GWAS_DIR, "ukb_wb_grin2a"))

## all results (tables, figures, log) are written under this folder
OUT_DIR     <- "results"
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)
OUT_PREFIX  <- file.path(OUT_DIR, "coloc_replication")

GWAS_N     <- 114400L                 # UKB1 total (cases + controls)
GWAS_S     <- 1268 / 114400           # case fraction
MIN_SNPS   <- 20L                     # minimum usable variants to attempt SuSiE / coloc

## PRIMARY analysis uses beta1 (the TMLE risk difference) for every locus.
## (chr12's lead p is beta2-driven; set BETA2_CHROMS <- c(12) to reproduce.)
BETA2_CHROMS        <- c()
BETA2_REQUIRE_JOINT <- TRUE           # on a beta2 chrom, DROP variants lacking a beta2 estimate
                                      # (keeps the locus on one contrast); FALSE = fall back to beta1
LOG_FILE   <- NULL                    # set in the run cell; captures per-dataset status


In [ ]:
## ---- unpack the bundled summary-statistic archives ----------------------
## The GWAS and eQTL summary statistics ship as zip files
# and are unzipped into data/ on first run.
## -- they are individual-level data; provide your own under data/ld_panels/ 
## (see the overview).
dir_mb <- function(path) if (!dir.exists(path)) 0 else
  sum(file.info(list.files(path, recursive = TRUE, full.names = TRUE))$size, na.rm = TRUE) / 1024^2

unpack <- function(zip, exdir) {
  if (!file.exists(zip)) { message("archive not found (skipping): ", zip); return(invisible()) }
  dir.create(exdir, showWarnings = FALSE, recursive = TRUE)
  utils::unzip(zip, exdir = exdir)
}
unpack(file.path(DATA_DIR, "gwas.zip"), file.path(DATA_DIR, "gwas"))
unpack(file.path(DATA_DIR, "eqtl.zip"), file.path(DATA_DIR, "eqtl"))

cat(sprintf("Extracted summary statistics -> GWAS %.1f MB | eQTL %.1f MB | total %.1f MB\n",
            dir_mb(file.path(DATA_DIR, "gwas")), dir_mb(file.path(DATA_DIR, "eqtl")),
            dir_mb(file.path(DATA_DIR, "gwas")) + dir_mb(file.path(DATA_DIR, "eqtl"))))



## 2. GWAS summary statistics & reference-panel LD

Fine-mapping and colocalisation both need (i) a signed z-score per variant and (ii) an
LD (correlation) matrix with concordant allele orientation. This section builds both.

- **Small helpers** — allele complement, minor/major allele from genotype counts, and a
  logger that mirrors messages to a run log.
- **GWAS orientation** — orient every variant's effect to its **minor allele** (the TMLE
  effect allele; see the notes above) and compute `z = beta / se`. We also record the
  allele set of each variant in both LD panels so that a variant is only used when it is
  present, with matching alleles, in the panels needed downstream.
- **`build_LD()`** — for a set of variants, use PLINK 2 to force the effect allele as the
  reference allele and compute the correlation matrix, so the LD sign matches the
  effect-allele-oriented z-scores. This function is reused by every analysis below.


In [ ]:
comp <- c(A = "T", T = "A", C = "G", G = "C")
flipb <- function(a) unname(comp[a])
is_ambig <- function(a, b) !is.na(a) & !is.na(b) & a %in% names(comp) & b %in% names(comp) & (comp[a] == b)

## minor / major allele from pooled genotype-count strings "A/A:120 A/G:6562 G/G:104556"
minor_major <- function(ctrl, case) {
  ac <- integer(0)
  for (s in c(ctrl, case)) {
    if (is.na(s) || s == "") next
    for (tok in strsplit(trimws(s), "\\s+")[[1]]) {
      kv <- strsplit(tok, ":")[[1]]; gt <- strsplit(kv[1], "/")[[1]]; n <- as.integer(kv[2])
      for (al in gt) ac[al] <- if (is.na(ac[al] %||% NA)) n else ac[al] + n
    }
  }
  if (length(ac) < 2) return(c(NA_character_, NA_character_))
  o <- names(sort(ac))                      # ascending count -> minor first
  c(o[1], o[length(o)])                      # (minor = effect allele, major)
}
`%||%` <- function(x, y) if (length(x) == 0 || is.na(x)) y else x

## log to console AND to LOG_FILE (so skip/failure reasons are captured)
logmsg <- function(fmt, ...) {
  msg <- sprintf(fmt, ...); cat(msg)
  if (exists("LOG_FILE") && !is.null(LOG_FILE)) cat(msg, file = LOG_FILE, append = TRUE)
}


In [ ]:
## ---- 1. GWAS: derive minor-allele (effect) orientation + z --------------
## Per-variant transition: variants on BETA2_CHROMS (chr12) use beta2 -- the
## transition that drives that locus's signal -- where beta2 is available; all
## other loci use beta1. Effect allele is the MINOR allele either way, so the
## harmonisation/orientation logic is unchanged by the transition choice. This
## applies automatically to EVERY chr12 eQTL file (GTEx, Jerber, MetaBrain) via
## the rsID merge, since the chr12 GWAS variants carry the beta2-derived z.
B1 <- "\u03b2\u2081"; S1 <- "se\u2081"; B2 <- "\u03b2\u2082"; S2 <- "se\u2082"   # beta1 se1 beta2 se2
g  <- fread(GWAS_FILE)
mm <- t(mapply(minor_major, g$ctrl_geno_counts, g$case_geno_counts))
g[, `:=`(EA = mm[, 1], OA = mm[, 2])]        # EA = minor (TMLE effect allele), OA = major
g <- g[!is.na(EA) & !is.na(OA) & EA %in% names(comp) & OA %in% names(comp)]

b1v <- g[[B1]]; s1v <- g[[S1]]; b2v <- g[[B2]]; s2v <- g[[S2]]
on_b2 <- g$chrom %in% BETA2_CHROMS
ok_b2 <- !is.na(b2v) & !is.na(s2v) & s2v > 0
keep  <- if (BETA2_REQUIRE_JOINT) !(on_b2 & !ok_b2) else rep(TRUE, nrow(g))
if (any(on_b2 & !ok_b2))
  cat(sprintf("beta2 chroms: %d variants lack beta2 -> %s\n",
              sum(on_b2 & !ok_b2), if (BETA2_REQUIRE_JOINT) "DROPPED" else "fall back to beta1"))
g <- g[keep]; b1v <- b1v[keep]; s1v <- s1v[keep]; b2v <- b2v[keep]; s2v <- s2v[keep]
on_b2 <- on_b2[keep]; ok_b2 <- ok_b2[keep]
use_b2 <- on_b2 & ok_b2
g[, `:=`(gbeta = fifelse(use_b2, b2v, b1v),
         gse   = fifelse(use_b2, s2v, s1v),
         transition = fifelse(use_b2, "beta2", "beta1"))]
g <- g[!is.na(gbeta) & !is.na(gse) & gse > 0]
g[, z := gbeta / gse]                        # z oriented to EA = minor allele
gwas <- g[, .(rsid = rsID, gEA = EA, gOA = OA, z, gbeta, gse, transition, chrom)]
gwas <- gwas[!duplicated(rsid)]
cat(sprintf("GWAS variants usable: %d (beta2: %d, beta1: %d)\n",
            nrow(gwas), sum(gwas$transition == "beta2"), sum(gwas$transition == "beta1")))

## Allele-set lookups for BOTH LD panels. A variant is only usable if it is
## present with matching alleles in the UKB panel (GWAS side) AND the 1000G panel
## (eQTL side), so that both fine-mappings run on the same variant set.
read_aset <- function(bf) {
  b <- fread(paste0(bf, ".bim"), header = FALSE, select = c(2, 5, 6), col.names = c("rsid", "bA1", "bA2"))
  b[, aset := mapply(function(x, y) paste(sort(c(toupper(x), toupper(y))), collapse = "/"), bA1, bA2)]
  setNames(b$aset, b$rsid)
}
BIMSET_UKB <- do.call(c, unname(lapply(KG_BFILES, read_aset)))          # UKB panels (per chrom), GWAS LD
BIMSET_UKB <- BIMSET_UKB[!duplicated(names(BIMSET_UKB))]
BIMSET_KG  <- read_aset(KG_EQTL)                                # 1000G EUR, eQTL LD

In [ ]:
## ---- LD from the per-locus UKB white-British panel (PLINK 2). REF is forced
## 'ref-based' sign is requested, so r is the correlation of effect-allele
## dosages -- matching the effect-allele-oriented z-scores. -----------------
build_LD <- function(snps, eff_by_snp, bfile) {
  tmp <- tempfile()
  fwrite(data.table(snps), paste0(tmp, ".snps"), col.names = FALSE)
  fwrite(data.table(snps, eff_by_snp[snps]), paste0(tmp, ".ref"), sep = "\t", col.names = FALSE)

  ## STEP 1: extract + set REF = effect allele, writing a new bfile
  cmd1 <- sprintf(paste("%s --bfile %s --extract %s.snps --make-founders",
                        "--ref-allele force %s.ref 2 1 --make-bed --out %s.eff"),
                  PLINK2, bfile, tmp, tmp, tmp)
  ## STEP 2: LD on that bfile; ref-based sign => correlation of REF(=effect) dosages
  cmd2 <- sprintf("%s --bfile %s.eff --r-unphased square ref-based --out %s",
                  PLINK2, tmp, tmp)

  logf <- paste0(tmp, ".log")
  for (cmd in c(cmd1, cmd2)) {
    st <- system(cmd, ignore.stdout = FALSE, ignore.stderr = FALSE)
    if (st != 0) {
      if (file.exists(logf)) cat("\n---- PLINK2 log ----\n", paste(readLines(logf), collapse = "\n"), "\n", sep = "")
      stop(sprintf("PLINK2 failed (exit %d). Command:\n%s", st, cmd))
    }
  }

  mfile <- paste0(tmp, ".unphased.vcor1"); vfile <- paste0(mfile, ".vars")
  if (!file.exists(mfile) || !file.exists(vfile))
    stop("PLINK2 LD output missing:\n", paste(readLines(logf), collapse = "\n"))

  ord <- fread(vfile, header = FALSE)[[1]]
  R <- as.matrix(fread(mfile, header = FALSE)); dimnames(R) <- list(ord, ord); R[is.na(R)] <- 0
  R
}

## 3 · Colocalisation (GWAS × eQTL) with `coloc.susie`

**Question:** at each locus, is the ME/CFS association driven by the *same* causal
variant as a nearby gene's expression (eQTL)? A shared causal variant is evidence that
the GWAS signal acts by changing that gene's expression.

**Method:** for every prepared eQTL dataset we (1) match and orient variants to a common
allele coding, (2) fit SuSiE separately to the GWAS and the eQTL signals (each using its
own matched LD panel), and (3) run `coloc.susie`, which compares the two sets of credible
sets and returns posterior probabilities for five hypotheses. The key quantity is
**PP.H4** — the posterior probability that the two traits **share** a causal variant
(as opposed to having distinct nearby signals, PP.H3).

The next two cells define the per-dataset routine and then run it across all eQTL files.
Every dataset returns a status row (success, or the reason it produced nothing), results
are written to `results/`, and colocalising pairs are ranked by PP.H4.


In [ ]:
## ---- 2a. standardise each eQTL source to canonical columns --------------
standardise_eqtl <- function(e, tag) {
  nm <- names(e)
  if ("MetaBeta" %in% nm && "SNPEffectAllele" %in% nm) {              # MetaBrain (GRCh38)
    als <- tstrsplit(as.character(e$SNPAlleles), "/", fixed = TRUE)
    a1 <- toupper(als[[1]]); a2 <- toupper(als[[2]]); eff <- toupper(as.character(e$SNPEffectAllele))
    data.table(rsid = as.character(e$rsid), position = as.integer(e$SNPPos),
               alt = eff, ref = ifelse(eff == a1, a2, a1),
               beta = as.numeric(e$MetaBeta), se = as.numeric(e$MetaSE),
               maf = pmin(as.numeric(e$SNPEffectAlleleFreq), 1 - as.numeric(e$SNPEffectAlleleFreq)),
               e_n = as.integer(e$MetaPN), gene_id = sub("\\..*$", "", as.character(e$Gene)),
               tissue = "MetaBrain_cortex", src = "MetaBrain")
  } else if ("assessed_allele" %in% nm && "a1" %in% nm && "a2" %in% nm) {   # Jerber (GRCh37)
    a1 <- toupper(as.character(e$a1)); a2 <- toupper(as.character(e$a2)); eff <- toupper(as.character(e$assessed_allele))
    se_col <- if ("beta_se" %in% nm) e$beta_se else e$se
    pos_col <- if ("pos" %in% nm) e$pos else e$snp_position
    data.table(rsid = as.character(e$rsid), position = as.integer(pos_col),
               alt = eff, ref = ifelse(eff == a1, a2, a1),
               beta = as.numeric(e$beta), se = as.numeric(se_col), maf = as.numeric(e$maf),
               e_n = if ("n_samples" %in% nm) as.integer(e$n_samples) else NA_integer_,
               gene_id = if ("feature_id" %in% nm) as.character(e$feature_id) else NA_character_,
               tissue = tag, src = "Jerber")
  } else {                                                            # GTEx-prep (canonical)
    if (!"se" %in% nm)       { h <- intersect(c("beta_se","slope_se"), nm); if (length(h)) setnames(e, h[1], "se") }
    if (!"beta" %in% nm)     { h <- intersect(c("slope"), nm);              if (length(h)) setnames(e, h[1], "beta") }
    if (!"position" %in% nm) { h <- intersect(c("pos","bp"), nm);           if (length(h)) setnames(e, h[1], "position") }
    need <- c("rsid","ref","alt","beta","se","maf","position")
    if (!all(need %in% names(e))) return(NULL)
    en <- if ("an" %in% names(e)) as.integer(e$an/2) else if ("n_samples" %in% names(e)) as.integer(e$n_samples) else NA_integer_
    data.table(rsid = as.character(e$rsid), position = as.integer(e$position),
               alt = toupper(as.character(e$alt)), ref = toupper(as.character(e$ref)),
               beta = as.numeric(e$beta), se = as.numeric(e$se), maf = as.numeric(e$maf), e_n = en,
               gene_id = if ("gene_id" %in% names(e)) as.character(e$gene_id) else NA_character_,
               tissue  = if ("tissue"  %in% names(e)) as.character(e$tissue)  else tag, src = "GTEx")
  }
}

## helpers -----------------------------------------------------------------
aln <- function(gEA, gOA, eA, eR) {
  if (gEA == eA && gOA == eR) return(1); if (gEA == eR && gOA == eA) return(-1)
  if (flipb(gEA) == eA && flipb(gOA) == eR) return(1); if (flipb(gEA) == eR && flipb(gOA) == eA) return(-1)
  NA_real_
}
stat_row <- function(tag, status, n = NA_integer_, g = NA_integer_, e = NA_integer_)
  data.table(dataset = tag, status = status, n_coloc_snps = n, ncs_gwas = g, ncs_eqtl = e)
run_ss <- function(D, label, tag) {                    # returns susie obj or NULL, logs outcome
  s <- tryCatch(suppressWarnings(suppressMessages(runsusie(D))),
                error = function(e) { logmsg("[%s]   %s runsusie ERROR: %s\n", tag, label, conditionMessage(e)); NULL })
  if (is.null(s)) return(NULL)
  ncs <- tryCatch(length(s$sets$cs), error = function(e) 0L)
  logmsg("[%s]   %s SuSiE: %d credible set(s)\n", tag, label, ncs)
  if (ncs == 0) NULL else s
}

## ---- 2b. process one eQTL dataset --------------------------------------
## extract credible-set variants (rsid + PIP) from a runsusie fit, for the combined CS table
extract_cs_susie <- function(s, side, tag, gene, tis, m) {
  if (is.null(s) || is.null(s$sets$cs) || !length(s$sets$cs)) return(NULL)
  pip  <- s$pip
  b    <- if (side == "gwas") m$sgn * m$gbeta else m$e_beta      # side-appropriate effect/SE from aligned m
  se   <- if (side == "gwas") m$gse           else m$e_se
  stat <- data.table(rsid = m$rsid, pos = m$position, beta = b, se = se)
  rbindlist(lapply(seq_along(s$sets$cs), function(k) {
    ids <- names(pip)[s$sets$cs[[k]]]
    d <- merge(data.table(rsid = ids, pip = round(as.numeric(pip[ids]), 4)), stat, by = "rsid", all.x = TRUE, sort = FALSE)
    d[, z := beta / se][, neglog10p := -(log10(2) + pnorm(-abs(z), log.p = TRUE) / log(10))]
    d[, `:=`(analysis = "coloc", side = side, locus = gene, dataset = tag, gene = gene, tissue = tis,
             credible_set = k, is_lead = pip == max(pip))]
    d[order(-pip), .(analysis, side, locus, dataset, gene, tissue, credible_set, rsid, pos,
                     beta = round(beta, 5), se = round(se, 5), z = round(z, 3), pip,
                     neglog10p = round(neglog10p, 2), is_lead)]
  }))
}

process <- function(path) {
  tag <- sub("\\.tsv$", "", basename(path))
  raw <- fread(path, sep = "\t", quote = "")   # force TAB: MetaBrain header has commas
  e <- standardise_eqtl(raw, tag)
  if (is.null(e)) { logmsg("[%s] SKIP unrecognised format | cols: %s\n", tag, paste(head(names(raw), 10), collapse = ",")); return(stat_row(tag, "skip:format")) }
  gene <- e$gene_id[1]; tis <- e$tissue[1]; src <- e$src[1]
  logmsg("\n[%s] src=%s gene=%s tissue=%s | %d rows\n", tag, src, gene, tis, nrow(e))
  e <- e[!is.na(rsid) & !is.na(beta) & !is.na(se) & se > 0 & !is.na(maf)]
  e[, `:=`(eEA = toupper(alt), eOA = toupper(ref))]
  e <- e[eEA %in% names(comp) & eOA %in% names(comp)][!duplicated(rsid)]

  m <- merge(gwas, e[, .(rsid, eEA, eOA, e_beta = beta, e_se = se, maf, position, e_n)], by = "rsid")
  n_merge <- nrow(m)
  m <- m[!is_ambig(gEA, gOA) & !is_ambig(eEA, eOA)]
  m[, sgn := mapply(aln, gEA, gOA, eEA, eOA)]; m <- m[!is.na(sgn)]
  n_align <- nrow(m)
  m[, aset := mapply(function(x, y) paste(sort(c(x, y)), collapse = "/"), eEA, eOA)]
  m <- m[rsid %in% names(BIMSET_UKB) & rsid %in% names(BIMSET_KG)][BIMSET_UKB[rsid] == aset & BIMSET_KG[rsid] == aset]
  n_ld <- nrow(m)
  logmsg("[%s] variants: shared_with_GWAS=%d  after_allele_align=%d  in_both_LD_panels=%d\n", tag, n_merge, n_align, n_ld)
  if (n_ld < MIN_SNPS) { logmsg("[%s] SKIP <%d usable variants\n", tag, MIN_SNPS); return(stat_row(tag, "skip:too_few_snps", n_ld)) }

  chr   <- as.character(m$chrom[1])
  bfile <- KG_BFILES[[chr]]
  if (is.null(bfile) || is.na(bfile)) { logmsg("[%s] no UKB LD panel for chr%s -- skip\n", tag, chr); return(stat_row(tag, paste0("skip:no_LD_panel_chr", chr), n_ld)) }
  effmap <- setNames(m$eEA, m$rsid)                       # REF <- effect allele (eQTL ALT) for BOTH panels
  R_g <- build_LD(m$rsid, effmap, bfile)                  # GWAS LD  <- UKB white-British panel
  R_e <- build_LD(m$rsid, effmap, KG_EQTL)               # eQTL LD  <- 1000G EUR (matches eQTL cohorts)
  if (is.null(R_g) || is.null(R_e)) return(stat_row(tag, "skip:LD_build_failed", n_ld))
  common <- m$rsid[m$rsid %in% intersect(rownames(R_g), rownames(R_e))]   # same variant set for both fits
  if (length(common) < MIN_SNPS) { logmsg("[%s] SKIP <%d variants shared across both LD panels\n", tag, MIN_SNPS); return(stat_row(tag, "skip:LD_panel_intersect", length(common))) }
  m <- m[rsid %in% common]; R_g <- R_g[m$rsid, m$rsid]; R_e <- R_e[m$rsid, m$rsid]
  trans <- paste(sort(unique(m$transition)), collapse = "/")
  eN <- if (all(is.na(m$e_n))) 200L else as.integer(median(m$e_n, na.rm = TRUE))
  logmsg("[%s] -> coloc: %d variants | transition=%s | eQTL_N~%d\n", tag, nrow(m), trans, eN)

  D_gwas <- list(beta = m$sgn * m$gbeta, varbeta = m$gse^2, snp = m$rsid, position = m$position, type = "cc", s = GWAS_S, N = GWAS_N, LD = R_g)
  D_eqtl <- list(beta = m$e_beta, varbeta = m$e_se^2, snp = m$rsid, position = m$position, type = "quant", N = eN, MAF = pmax(pmin(m$maf, 0.5), 1e-4), LD = R_e)
  s1 <- run_ss(D_gwas, "GWAS", tag); s2 <- run_ss(D_eqtl, "eQTL", tag)
  ncs_g <- if (is.null(s1)) 0L else length(s1$sets$cs); ncs_e <- if (is.null(s2)) 0L else length(s2$sets$cs)
  ## record EVERY credible set found on either trait (independent of whether coloc runs)
  if (exists(".cs_coloc_acc")) {
    cc <- rbindlist(list(extract_cs_susie(s1, "gwas", tag, gene, tis, m),
                         extract_cs_susie(s2, "eqtl", tag, gene, tis, m)), fill = TRUE)
    if (!is.null(cc) && nrow(cc)) .cs_coloc_acc[[length(.cs_coloc_acc) + 1L]] <<- cc
  }
  if (is.null(s1) || is.null(s2)) {
    logmsg("[%s] STOP no credible set (GWAS=%d, eQTL=%d) -> cannot coloc\n", tag, ncs_g, ncs_e)
    return(stat_row(tag, sprintf("no_credible_set(gwas=%d,eqtl=%d)", ncs_g, ncs_e), nrow(m), ncs_g, ncs_e))
  }
  cs <- tryCatch(suppressMessages(coloc.susie(s1, s2)), error = function(x) { logmsg("[%s] coloc.susie ERROR: %s\n", tag, conditionMessage(x)); NULL })
  if (is.null(cs) || is.null(cs$summary) || nrow(cs$summary) == 0) { logmsg("[%s] STOP coloc.susie returned no signal pair\n", tag); return(stat_row(tag, "no_coloc_pair", nrow(m), ncs_g, ncs_e)) }

  out <- as.data.table(cs$summary)
  out[, `:=`(dataset = tag, gene = gene, tissue = tis, n_coloc_snps = nrow(m),
             transition = trans, ncs_gwas = ncs_g, ncs_eqtl = ncs_e, status = "ok")]
  fwrite(out, sprintf("%s_%s.tsv", OUT_PREFIX, tag), sep = "\t")
  logmsg("[%s] DONE %d signal-pair(s) | best PP.H4=%.3f\n", tag, nrow(out), max(out$PP.H4.abf))
  out
}


In [ ]:
## ---- 3. run all datasets; every dataset returns a status row ------------
LOG_FILE <- sprintf("%s_run.log", OUT_PREFIX)
cat("", file = LOG_FILE)
files <- list.files(EQTL_DIR, pattern = "\\.tsv$", full.names = TRUE)
logmsg("Found %d eQTL datasets in %s\n", length(files), EQTL_DIR)

.cs_coloc_acc <- list()                          # collect coloc credible sets across datasets
res <- rbindlist(lapply(files, process), fill = TRUE)
CS_COLOC <- if (length(.cs_coloc_acc)) rbindlist(.cs_coloc_acc, fill = TRUE) else NULL

## status overview for EVERY dataset (ok or reason it produced nothing)
logmsg("\n================ STATUS (all datasets) ================\n")
ov <- unique(res[, .(dataset,
                     status = if ("status" %in% names(res)) status else NA_character_,
                     n_coloc_snps,
                     ncs_gwas = if ("ncs_gwas" %in% names(res)) ncs_gwas else NA_integer_,
                     ncs_eqtl = if ("ncs_eqtl" %in% names(res)) ncs_eqtl else NA_integer_,
                     bestH4 = if ("PP.H4.abf" %in% names(res)) round(PP.H4.abf, 3) else NA_real_)],
             by = "dataset")
print(ov)

## coloc hits only, ranked
if ("PP.H4.abf" %in% names(res)) {
  succ <- res[!is.na(PP.H4.abf)]
  if (nrow(succ)) {
    setorder(succ, -PP.H4.abf)
    fwrite(succ, sprintf("%s_SUMMARY.tsv", OUT_PREFIX), sep = "\t")
    cat("\n==== coloc results (ranked by PP.H4) ====\n")
    print(succ[, .(dataset, gene, tissue, transition, n_coloc_snps, PP.H4.abf, PP.H3.abf, PP.H0.abf, hit1, hit2)])
  } else cat("\nNo coloc H4 produced -- see status above and", LOG_FILE, "for per-stage detail.\n")
}
cat(sprintf("\nFull per-stage log: %s\n", LOG_FILE))


## 4 · Fine-mapping the GWAS signal with SuSiE

SuSiE fine-maps the GWAS signal alone (using the matched UKB white-British LD
panel). It returns, for each independent signal, a **95% credible set** which is the smallest
set of variants that is 95% likely to contain the causal variant. This is marked by a **posterior
inclusion probability (PIP)** per variant. By default a credible set is only reported if
it passes a *purity* filter (minimum pairwise |r| ≥ 0.5); loci whose signal is too weak
or too diffuse to localise therefore return no 95% credible set.

The cells below define:
- helper utilities for the figures — GRCh37→GRCh38 liftover and protein-coding gene models
  via the Ensembl REST API and plotting utilities;
- `finemap_full_gwas()` / `plot_finemap()` — run SuSiE per locus and draw the three-panel
  figure (association, PIP, gene track);
- the run loop over all loci, which also writes a combined credible-set table.


In [ ]:
## ---- rsID -> hg19 position lookup (for the fine-map tables and x-axes) ----
gpos <- fread(GWAS_FILE, select = c("rsID", "pos")); setnames(gpos, "rsID", "rsid")

## ---- hg38 axis + gene track via Ensembl REST (jsonlite only; no Bioconductor) ----
## Needs only jsonlite + network access to rest.ensembl.org (no chain file, no compiled pkgs).
suppressWarnings(suppressMessages({
  if (!requireNamespace("jsonlite", quietly = TRUE))
    install.packages("jsonlite", repos = "https://cloud.r-project.org")
}))

.ens_get <- function(path) {
  url <- paste0("https://rest.ensembl.org", path,
                if (grepl("\\?", path)) "&" else "?", "content-type=application/json")
  txt <- tryCatch(readLines(url, warn = FALSE), error = function(e) NULL)
  if (is.null(txt)) return(NULL)
  tryCatch(jsonlite::fromJSON(paste(txt, collapse = "")), error = function(e) NULL)
}

## liftover GRCh37 -> GRCh38 for positions on one chromosome, reduced to a constant
## region offset. Degrades to the GRCh37 position (x-axis unchanged) on any failure.
lift38 <- function(chr, pos19) {
  pos19 <- as.integer(pos19)
  lo <- min(pos19); hi <- max(pos19)
  r   <- .ens_get(sprintf("/map/human/GRCh37/%s:%d..%d:1/GRCh38", chr, lo, hi))
  off <- tryCatch({
    mp <- r$mappings                       # nested $mapped / $original frames
    as.numeric(mp$mapped$start)[1] - as.numeric(mp$original$start)[1]
  }, error = function(e) NA_real_)
  if (length(off) != 1 || !is.finite(off)) {
    message("[finemap] Ensembl liftover failed for chr", chr, " -- x-axis stays GRCh37")
    return(as.numeric(pos19))
  }
  message(sprintf("[finemap] chr%s GRCh37->GRCh38 offset = %+.0f bp", chr, off))
  as.numeric(pos19) + off
}

## protein-coding genes (+exons) overlapping a GRCh38 window, via /overlap/region
.have_genes <- requireNamespace("jsonlite", quietly = TRUE)
genes_hg38 <- function(chr, start38, end38) {
  if (is.na(start38) || is.na(end38)) return(list())
  g <- .ens_get(sprintf("/overlap/region/human/%s:%d-%d?feature=gene", chr, as.integer(start38), as.integer(end38)))
  if (is.null(g) || length(g) == 0) return(list())
  g <- as.data.frame(g); g <- g[g$biotype == "protein_coding" & !is.na(g$biotype), , drop = FALSE]
  if (!nrow(g)) return(list())
  ex <- .ens_get(sprintf("/overlap/region/human/%s:%d-%d?feature=exon", chr, as.integer(start38), as.integer(end38)))
  ex <- if (!is.null(ex) && length(ex)) as.data.frame(ex) else NULL
  canon <- sub("\\..*$", "", g$canonical_transcript)          # "ENST...6" -> "ENST..."
  expar <- if (!is.null(ex)) sub("\\..*$", "", ex$Parent) else character(0)
  lapply(seq_len(nrow(g)), function(i) {
    sym <- g$external_name[i]; if (is.na(sym) || sym == "") sym <- g$gene_id[i]
    exi <- if (!is.null(ex)) ex[expar == canon[i], c("start", "end"), drop = FALSE] else NULL
    list(symbol = sym, start = g$start[i], end = g$end[i],
         strand = ifelse(g$strand[i] > 0, "+", "-"), exons = exi)
  })
}


In [ ]:
## ============================ plotting settings ====================
GENE_CEX <- 1.55; AX_CEX <- 2.00; LAB_CEX <- 2.30      # gene labels / ticks / axis titles
PT_CEX   <- 1.90; DIA_CEX <- 3.20; ANN_CEX <- 1.50     # points / highlight diamonds / annotations
LEG_CEX  <- 1.85; PNG_W <- 1850L; PNG_RES <- 150L; UNIT_PX <- 776

.gene_label <- function(g) sprintf("%s%s", g$symbol, if (g$strand == "+") " \u2192" else " \u2190")

## gene-track layout: pack genes into rows using LABEL width 
gene_layout <- function(chr, xr38, cex.gene = GENE_CEX) {
  gl <- genes_hg38(chr, xr38[1], xr38[2])
  if (length(gl)) gl <- gl[vapply(gl, function(g) !grepl("^ENSG[0-9]+$", g$symbol), logical(1))]
  if (!length(gl)) return(list(gl = list(), row = integer(0), nr = 0L))
  gl <- gl[order(vapply(gl, function(g) g$start, numeric(1)))]
  labs <- vapply(gl, .gene_label, character(1))
  lw <- tryCatch({
    pdf(NULL, width = PNG_W / PNG_RES, height = 2.4)
    par(mar = c(6.0, 7.4, 0.5, 2.0))
    plot(NA, xlim = xr38 / 1e6, ylim = c(0, 1), axes = FALSE, xlab = "", ylab = "")
    w <- strwidth(labs, cex = cex.gene, font = 3) * 1.06     # 6% safety margin
    dev.off(); w
  }, error = function(e) { try(dev.off(), silent = TRUE); rep(0.05 * diff(xr38) / 1e6, length(labs)) })
  lw <- pmin(lw, 0.90 * diff(xr38) / 1e6)                    # never wider than the window
  st <- vapply(gl, function(g) g$start, numeric(1)) / 1e6
  en <- vapply(gl, function(g) g$end,   numeric(1)) / 1e6
  x1 <- pmax(st, xr38[1] / 1e6); x2 <- pmin(en, xr38[2] / 1e6)
  ctr <- (x1 + x2) / 2
  ctr <- pmin(pmax(ctr, xr38[1] / 1e6 + lw / 2), xr38[2] / 1e6 - lw / 2)   # keep labels in-panel
  left <- pmin(x1, ctr - lw / 2); right <- pmax(x2, ctr + lw / 2)          # body U label extent
  pad <- 0.012 * diff(xr38) / 1e6
  rowend <- numeric(0); row <- integer(length(gl))
  for (i in seq_along(gl)) {
    r <- which(left[i] > rowend + pad)[1]
    if (is.na(r)) { rowend <- c(rowend, right[i]); row[i] <- length(rowend) }
    else          { rowend[r] <- right[i];         row[i] <- r }
  }
  list(gl = gl, row = row, nr = max(row), ctr = ctr, x1 = x1, x2 = x2)
}

## draw a gene track (exons = filled boxes, introns/body = line) on an hg38 Mb axis
draw_gene_track <- function(chr, xr38, lay = NULL, cex.gene = GENE_CEX,
                            cex.lab = LAB_CEX, cex.axis = AX_CEX) {
  if (is.null(lay)) lay <- gene_layout(chr, xr38, cex.gene)
  nr <- max(lay$nr, 1L)
  plot(NA, xlim = xr38 / 1e6, ylim = c(0, 1),
       xlab = sprintf("Chromosome %d position (Mb, GRCh38)", chr),
       ylab = "", yaxt = "n", bty = "n", cex.axis = cex.axis, cex.lab = cex.lab)
  if (!length(lay$gl)) {
    text(mean(xr38 / 1e6), 0.5, "No protein-coding genes in window", col = "grey55", cex = 1.7, font = 3)
    return(invisible())
  }
  yof <- function(r) 1 - (r - 0.5) / nr; eh <- 0.26 / nr
  for (i in seq_along(lay$gl)) {
    g <- lay$gl[[i]]; y <- yof(lay$row[i])
    segments(lay$x1[i], y, lay$x2[i], y, col = "grey45", lwd = 2.4)
    if (!is.null(g$exons) && nrow(g$exons))
      rect(g$exons$start / 1e6, y - eh, g$exons$end / 1e6, y + eh, col = "#2c3e50", border = NA)
    text(lay$ctr[i], y + eh + 0.22 / nr, .gene_label(g), cex = cex.gene, font = 3, xpd = TRUE)
  }
}

## ---- credible-set + LD (r^2) colour palettes and shared panel prep ----
SETPAL <- c("#7b3fa0", "#1a9641", "#e6550d", "#2c7fb8", "#d7301f", "#b15928")   # one colour per credible set
setcol <- function(k) SETPAL[((k - 1) %% length(SETPAL)) + 1]

LDPAL <- c("#3f6fb0", "#67c2d4", "#5cbf5c", "#f0a83c", "#e03030")     # r^2 <0.2 .. >0.8
LDBRK <- c(-Inf, .2, .4, .6, .8, Inf)
LDLEG <- c("0.8\u20131.0", "0.6\u20130.8", "0.4\u20130.6", "0.2\u20130.4", "< 0.2")

.ld_legend <- function(title_expr, cex = LEG_CEX) {
  usr <- par("usr")
  x0 <- usr[1] + 0.012 * diff(usr[1:2])
  y0 <- usr[4] - 0.015 * diff(usr[3:4])
  text(x0, y0, title_expr, adj = c(0, 1), cex = cex, xpd = FALSE)
  th <- strheight("Mg", cex = cex) * 2.0
  legend(x = x0, y = y0 - th, xjust = 0, yjust = 1,
         legend = LDLEG, pt.bg = rev(LDPAL), pch = 21, pt.cex = 2.1,
         cex = cex, bty = "n", adj = 0, x.intersp = 0.9, y.intersp = 1.05, xpd = FALSE)
}

## copy and colour by r^2 to an anchor variant, build the x-axis
.prep_panel <- function(m, R, chr, anchor) {
  m <- copy(m)
  m[, r2 := as.numeric(R[anchor, rsid])^2]
  m[, col := LDPAL[cut(r2, LDBRK, labels = FALSE)]]
  if (!"pos38" %in% names(m)) m[, pos38 := lift38(chr, pos)]
  usex <- if (all(is.na(m$pos38))) "pos" else "pos38"
  m2 <- m[!is.na(get(usex))]; m2[, xx := get(usex) / 1e6]
  list(m2 = m2, usex = usex, build_lab = if (usex == "pos38") "GRCh38" else "GRCh37",
       xr = range(m2$xx), xr38 = range(m2[[usex]]))
}


In [ ]:
## ---- fine-map on the FULL GWAS variant set in the region (UKB panel only, no eQTL) ----
finemap_full_gwas <- function(name, chr, bfile) {
  if (!file.exists(paste0(bfile, ".bed"))) { logmsg("[finemap %s] panel bed not found: %s\n", name, basename(bfile)); return(NULL) }
  aset_p <- read_aset(bfile)                                        # this locus's panel allele-sets define the window
  m <- gwas[chrom == chr]
  m[, aset := mapply(function(x, y) paste(sort(c(x, y)), collapse = "/"), gEA, gOA)]
  m <- m[rsid %in% names(aset_p)][aset_p[rsid] == aset]
  if (nrow(m) < MIN_SNPS) { logmsg("[finemap %s] < %d variants after panel match -- skip\n", name, MIN_SNPS); return(NULL) }
  R <- build_LD(m$rsid, setNames(m$gEA, m$rsid), bfile); if (is.null(R)) return(NULL)   # LD REF = minor (GWAS effect) allele
  m <- m[rsid %in% rownames(R)]; R <- R[m$rsid, m$rsid]
  D <- list(beta = m$gbeta, varbeta = m$gse^2, snp = m$rsid, type = "cc", s = GWAS_S, N = GWAS_N, LD = R)
  .finemap_finish(m, R, D, as.integer(chr), sprintf("full GWAS | %s", name), tag = name)
}

## run SuSiE, tag credible sets, write per-variant + credible-set tables, return the fit bundle
.finemap_finish <- function(m, R, D, chr, what = "", tag = sprintf("chr%d", chr)) {
  fit <- tryCatch(suppressWarnings(suppressMessages(runsusie(D))), error = function(e) NULL); if (is.null(fit)) return(NULL)
  cs <- fit$sets$cs
  cs_ids <- if (length(cs)) lapply(cs, function(ix) names(fit$pip)[ix]) else list()   # CS rsIDs before unnaming pip
  m[, pip := as.numeric(fit$pip[rsid])]
  m[, neglog10p := -(log10(2) + pnorm(-abs(z), log.p = TRUE) / log(10))]
  m[, csn := NA_integer_]; for (k in seq_along(cs_ids)) m[rsid %in% cs_ids[[k]], csn := k]
  m <- merge(m, gpos, by = "rsid")
  m[, pos38 := lift38(chr, pos)]
  logmsg("[finemap chr%d | %s] %d variants, %d credible set(s), csn tagged=%d\n", chr, what, nrow(m), length(cs), sum(!is.na(m$csn)))
  fwrite(m[order(-pip), .(rsid, pos_hg19 = pos, pos_hg38 = pos38, z, neglog10p, pip, csn)], sprintf("%s_gwas_finemap_%s.tsv", OUT_PREFIX, tag), sep = "\t")
  ## credible-set-only table with PIPs (one row per CS variant), lead flagged per set
  cstab <- m[!is.na(csn)][order(csn, -pip),
                          .(analysis = "fine-mapping", side = "gwas", locus = tag, chr = chr, credible_set = csn,
                            rsid, pos_hg19 = pos, pos_hg38 = pos38, beta = round(gbeta, 5), se = round(gse, 5),
                            pip = round(pip, 4), z = round(z, 3), neglog10p = round(neglog10p, 2))]
  if (nrow(cstab)) cstab[, is_lead := pip == max(pip), by = credible_set]
  list(chr = chr, tag = tag, m = m, R = R, ncs = length(cs), cstab = cstab)
}

## ---- three-panel fine-map figure: association + PIP + gene track, on the GRCh38 axis ----
## All credible sets are highlighted (one colour each); the gene panel height scales with
## the number of packed gene rows so labels stay legible in dense regions.
plot_finemap <- function(fm) {
  m <- fm$m; R <- fm$R; chr <- fm$chr
  csidx <- sort(unique(na.omit(m$csn)))
  lead  <- m$rsid[which.max(m$pip)]                      # top-PIP variant anchors the r^2 scale
  P <- .prep_panel(m, R, chr, lead); m2 <- P$m2; xr <- P$xr; xr38 <- P$xr38
  set_lead <- setNames(vapply(csidx, function(k) m2[csn == k][which.max(pip), rsid], character(1)), csidx)
  labpos <- function(x) if (x > xr[1] + 0.82*diff(xr)) 2 else if (x < xr[1] + 0.18*diff(xr)) 4 else 3
  ymaxt <- max(m2$neglog10p, na.rm = TRUE) * 1.18 + 0.2
  ymaxp <- if (all(is.na(m2$pip))) 1 else min(1, max(m2$pip, na.rm = TRUE) * 1.22 + 0.05)
  add_genes <- isTRUE(.have_genes) && P$usex == "pos38"

  lay <- if (add_genes) gene_layout(chr, xr38) else NULL
  gene_px <- if (add_genes) 170 + 130 * max(lay$nr, 1L) else 0
  png(sprintf("%s_finemap_%s.png", OUT_PREFIX, fm$tag), width = PNG_W,
      height = round((1 + 1.04) * UNIT_PX + gene_px), res = PNG_RES)
  if (add_genes) layout(matrix(1:3, 3, 1), heights = c(1, 1.04, gene_px / UNIT_PX))
  else           layout(matrix(1:2, 2, 1), heights = c(1, 1.08))

  ## --- top: association (mark + label each credible set's lead) ---
  par(mar = c(0.8, 7.4, 1.6, 2.0))
  plot(m2$xx, m2$neglog10p, pch = 21, bg = m2$col, col = "grey35", cex = PT_CEX,
       xlim = xr, ylim = c(0, ymaxt), xaxt = "n", xlab = "",
       ylab = expression(-log[10](italic(P))), cex.axis = AX_CEX, cex.lab = LAB_CEX)
  for (k in csidx) { lr <- m2[rsid == set_lead[as.character(k)]]
    points(lr$xx, lr$neglog10p, pch = 23, bg = setcol(k), col = "black", cex = DIA_CEX, lwd = 2.2)
    text(lr$xx, lr$neglog10p, lr$rsid, pos = labpos(lr$xx), cex = ANN_CEX, font = 2, offset = 1.2) }
  .ld_legend(expression(paste(italic(r)^2, " to lead variant")))

  ## --- middle: PIP (every credible set in its own colour; each set's lead labelled) ---
  par(mar = if (add_genes) c(0.8, 7.4, 0.8, 2.0) else c(6.2, 7.4, 0.8, 2.0))
  plot(m2$xx, m2$pip, pch = 21, bg = m2$col, col = "grey35", cex = PT_CEX,
       xlim = xr, ylim = c(0, ymaxp), xaxt = if (add_genes) "n" else "s",
       xlab = if (add_genes) "" else sprintf("Chromosome %d position (Mb, %s)", chr, P$build_lab),
       ylab = "Posterior inclusion probability", cex.axis = AX_CEX, cex.lab = LAB_CEX)
  if (length(csidx)) {
    for (k in csidx) { sset <- m2[csn == k]
      points(sset$xx, sset$pip, pch = 5, col = setcol(k), cex = 2.5, lwd = 2.4)
      lr <- m2[rsid == set_lead[as.character(k)]]
      points(lr$xx, lr$pip, pch = 23, bg = setcol(k), col = "black", cex = DIA_CEX, lwd = 2.2)
      text(lr$xx, lr$pip, sprintf("%s (PIP = %.2f)", lr$rsid, lr$pip),
           pos = labpos(lr$xx), cex = ANN_CEX, font = 2, offset = 1.2) }
    legend("topright", legend = sprintf("Credible set %d (%d variants)", csidx,
             vapply(csidx, function(k) sum(m2$csn == k, na.rm = TRUE), integer(1))),
           pch = 5, col = vapply(csidx, setcol, character(1)), pt.cex = 2.1, pt.lwd = 2.4,
           cex = 1.45, bty = "n")
  } else text(mean(xr), ymaxp * 0.6, "No credible set identified", cex = 2.1, col = "grey45", font = 3)

  ## --- bottom: gene track (GRCh38) ---
  if (add_genes) { par(mar = c(6.0, 7.4, 0.5, 2.0)); draw_gene_track(chr, xr38, lay = lay) }
  dev.off()
  cat(sprintf("Wrote %s_finemap_%s.png (%s axis%s)\n", OUT_PREFIX, fm$tag, P$build_lab,
              if (add_genes) sprintf(" + gene track, %d row(s)", lay$nr) else ""))
}


In [ ]:
## Fine-map every locus the SAME way: full GWAS window + matched UKB white-British panel.
## Loci whose signal does not robustly fine-map return no significant credible set, 
## and plot_finemap draws the region as such.
## Two chr16 loci (grin2a, sbk1) are keyed by name so their outputs do not collide.
FINEMAP_LOCI <- list(
  etv5   = list(chr =  3L, bfile = file.path(LD_GWAS_DIR, "ukb_wb_etv5"),        rep = "rs115186419"),
  csmd1  = list(chr =  8L, bfile = file.path(LD_GWAS_DIR, "ukb_wb_csmd1"),       rep = "rs73175505"),
  bicd1  = list(chr = 12L, bfile = file.path(LD_GWAS_DIR, "ukb_wb_bicd1_dnm1l"), rep = "rs261902"),
  clybl  = list(chr = 13L, bfile = file.path(LD_GWAS_DIR, "ukb_wb_clybl"),       rep = "rs117553493"),
  rora   = list(chr = 15L, bfile = file.path(LD_GWAS_DIR, "ukb_wb_rora"),        rep = "rs72741654"),
  grin2a = list(chr = 16L, bfile = file.path(LD_GWAS_DIR, "ukb_wb_grin2a"),      rep = "rs74963073"),
  sbk1   = list(chr = 16L, bfile = file.path(LD_GWAS_DIR, "ukb_wb_sbk1"),        rep = "rs76847656"))

## pre-flight: is each replicated variant actually present in the GWAS and its LD panel?
cat("\n=== replicated-variant availability per locus ===\n")
for (nm in names(FINEMAP_LOCI)) {
  L <- FINEMAP_LOCI[[nm]]
  inG <- L$rep %in% gwas[chrom == L$chr, rsid]
  inP <- tryCatch(L$rep %in% names(read_aset(L$bfile)), error = function(e) NA)
  cat(sprintf("  %-7s chr%-2d %-12s  in GWAS: %-5s  in LD panel: %s\n", nm, L$chr, L$rep, inG, inP))
}

## fine-map each locus and write its 3-panel figure (assoc + PIP + gene track)
cs_all <- list()
for (nm in names(FINEMAP_LOCI)) {
  L  <- FINEMAP_LOCI[[nm]]
  fm <- finemap_full_gwas(nm, L$chr, L$bfile)
  if (is.null(fm)) { logmsg("[finemap %s] no fine-map produced (panel/variant issue) -- no figure\n", nm); next }
  plot_finemap(fm)
  if (!is.null(fm$cstab) && nrow(fm$cstab)) cs_all[[length(cs_all) + 1L]] <- fm$cstab
}

## ---- ONE combined credible-set table: fine-mapping + coloc, all loci --------
cs_parts <- list()
if (length(cs_all)) cs_parts[[length(cs_parts) + 1L]] <- rbindlist(cs_all, fill = TRUE)
if (exists("CS_COLOC") && !is.null(CS_COLOC) && nrow(CS_COLOC)) cs_parts[[length(cs_parts) + 1L]] <- CS_COLOC

if (length(cs_parts)) {
  CS_ALL <- rbindlist(cs_parts, fill = TRUE, use.names = TRUE)
  setcolorder(CS_ALL, intersect(c("analysis","side","locus","dataset","gene","tissue","chr","credible_set",
                                  "rsid","pos","pos_hg19","pos_hg38","beta","se","z","pip","neglog10p","is_lead"), names(CS_ALL)))
  setorder(CS_ALL, analysis, locus, credible_set, -pip)
  fwrite(CS_ALL, sprintf("%s_credible_sets_ALL.tsv", OUT_PREFIX), sep = "\t")
  cat(sprintf("\nWrote %s_credible_sets_ALL.tsv  (%d rows: %d fine-mapping, %d coloc)\n",
              OUT_PREFIX, nrow(CS_ALL), sum(CS_ALL$analysis == "fine-mapping"), sum(CS_ALL$analysis == "coloc")))
  print(CS_ALL)
}
